In [1]:
!pip install pygeohash catboost -q

import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print(train.shape)
print(test.shape)
print(sample_submission.head())

(77299, 11)
(41778, 10)
   Index    demand
0      0  0.090768
1      1  0.089885
2      2  0.007037
3      3  0.079087
4      4  0.054636


In [5]:
for df in [train, test]:
    time_parts = df['timestamp'].astype(str).str.split(':', expand=True)

    df['hour'] = time_parts[0].astype(int)
    df['minute'] = time_parts[1].astype(int)

    df['is_peak'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

In [6]:
for df in [train, test]:
    df['latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
    df['longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])

    df['time_slot'] = df['hour'].astype(str) + "_" + df['minute'].astype(str)
    df['road_hour'] = df['RoadType'].astype(str) + "_" + df['hour'].astype(str)
    df['road_vehicle'] = df['RoadType'].astype(str) + "_" + df['LargeVehicles'].astype(str)
    df['road_lane'] = df['RoadType'].astype(str) + "_" + df['NumberofLanes'].astype(str)
    df['geo_road'] = df['geohash'].astype(str) + "_" + df['RoadType'].astype(str)

In [7]:
geo_counts = train['geohash'].value_counts()

train['geo_freq'] = train['geohash'].map(geo_counts)
test['geo_freq'] = test['geohash'].map(geo_counts).fillna(0)

In [8]:
features = [
    'geohash',
    'day',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'hour',
    'minute',
    'is_peak',
    'time_slot',
    'geo_freq',
    'road_hour',
    'road_vehicle',
    'road_lane',
    'geo_road',
    'latitude',
    'longitude'
]

cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'time_slot',
    'road_hour',
    'road_vehicle',
    'road_lane',
    'geo_road'
]

print(train[features].head())
print(test[features].head())

  geohash  day     RoadType  NumberofLanes LargeVehicles Landmarks  \
0  qp02z1   48          NaN              1   Not Allowed        No   
1  qp02zt   48  Residential              3       Allowed       Yes   
2  qp08bj   48  Residential              1   Not Allowed        No   
3  qp08gt   48  Residential              1   Not Allowed        No   
4  qp02zq   48  Residential              1   Not Allowed        No   

   Temperature Weather  hour  minute  is_peak time_slot  geo_freq  \
0          NaN     NaN     0       0        0       0_0        33   
1    31.104565   Sunny     0       0        0       0_0        89   
2    25.919267   Sunny     0       0        0       0_0        67   
3          NaN   Rainy     0       0        0       0_0        42   
4    10.803667   Rainy     0       0        0       0_0        36   

       road_hour             road_vehicle      road_lane            geo_road  \
0          nan_0          nan_Not Allowed          nan_1          qp02z1_nan   
1  R

In [10]:
# Check missing values in final features
print(train[features].isna().sum())
print(test[features].isna().sum())

geohash             0
day                 0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
hour                0
minute              0
is_peak             0
time_slot           0
geo_freq            0
road_hour           0
road_vehicle        0
road_lane           0
geo_road            0
latitude            0
longitude           0
dtype: int64
geohash             0
day                 0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
hour                0
minute              0
is_peak             0
time_slot           0
geo_freq            0
road_hour           0
road_vehicle        0
road_lane           0
geo_road            0
latitude            0
longitude           0
dtype: int64


In [11]:
# Fill categorical missing values
for col in cat_features:
    train[col] = train[col].fillna("Unknown").astype(str)
    test[col] = test[col].fillna("Unknown").astype(str)

# Fill numeric missing values
num_features = [f for f in features if f not in cat_features]

for col in num_features:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(train[col].median())

print("Train missing:", train[features].isna().sum().sum())
print("Test missing:", test[features].isna().sum().sum())

Train missing: 0
Test missing: 0


In [12]:
X_full = train[features]
y_full = train['demand']
X_test = test[features]

final_model = CatBoostRegressor(
    iterations=2000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=200
)

final_model.fit(
    X_full,
    y_full,
    cat_features=cat_features
)

0:	learn: 0.0766849	total: 257ms	remaining: 8m 33s
200:	learn: 0.9333579	total: 26.6s	remaining: 3m 57s
400:	learn: 0.9455445	total: 54.4s	remaining: 3m 36s
600:	learn: 0.9519432	total: 1m 22s	remaining: 3m 13s
800:	learn: 0.9557225	total: 1m 51s	remaining: 2m 47s
1000:	learn: 0.9586497	total: 2m 21s	remaining: 2m 21s
1200:	learn: 0.9607821	total: 2m 51s	remaining: 1m 53s
1400:	learn: 0.9628502	total: 3m 19s	remaining: 1m 25s
1600:	learn: 0.9645958	total: 3m 48s	remaining: 57s
1800:	learn: 0.9661766	total: 4m 17s	remaining: 28.5s
1999:	learn: 0.9674688	total: 4m 47s	remaining: 0us


CatBoostRegressor(depth=8, eval_metric='R2', iterations=2000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=200)

In [13]:
test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_pred
})

submission.to_csv("final_submission_91008_clean.csv", index=False)

print(submission.head())
print(submission.shape)

   Index    demand
0      0  0.044225
1      1  0.028439
2      2  0.021086
3      3  0.031300
4      4  0.063100
(41778, 2)
